# L04 · 数据清洗 + 复权 + 多股对齐

**预计时长**：75–90 min | **难度**：⭐⭐⭐⭐ | **前置**：L03

## 本节目标
1. 理解除权除息（送股 / 转股 / 分红）对 K 线的影响
2. 区分前复权（qfq）/ 后复权（hfq）/ 不复权（""）
3. 多股数据对齐：`pd.concat` / `reindex` / 交易日历
4. 写出能复用的 panel 数据生成函数，缓存到 parquet 供 L05+ 使用

> ⚠️ 这是 Phase 1 最硬核的一节。L05/L06/L10 全部依赖本节产出的对齐数据。

## 第 1 段：金融概念

### 除权除息（简称"除权"）
公司分红送股那天，股价会**人为下调**。例：
- 比亚迪分红 10 派 10 元（每股分 1 元）
- 股权登记日收盘价 250 元
- 除权日开盘价自动变成 249 元（250 - 1）

如果你用**不复权**数据算收益率，除权日会显示"-0.4%"，但股东实际没亏（收到 1 元现金）。

### 前复权 vs 后复权 vs 不复权
| 类型 | akshare 参数 | 特征 | 用途 |
|------|-------------|------|------|
| 不复权 | `adjust=""` | 真实成交价，有跳空 | 看"当时真实价格" |
| 前复权 | `adjust="qfq"` | 以**最新价**为基准，往历史扣减 | 量化**最常用**，看历史涨跌"平滑" |
| 后复权 | `adjust="hfq"` | 以**最早价**为基准，往后累加 | 长期持有收益率计算 |

**新手陷阱**：所有量化策略回测都用 **qfq**，否则除权日会触发虚假信号。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

## 第 2 段：用比亚迪实测复权差异

同时拉三份不同 adjust 的比亚迪数据，看除权日的跳空差异。

In [ ]:
# 拉三种复权方式
byd_raw  = get_stock_data("002594", adjust="",   force_refresh=False)
byd_qfq  = get_stock_data("002594", adjust="qfq", force_refresh=False)
byd_hfq  = get_stock_data("002594", adjust="hfq", force_refresh=False)

# 找一个明显的除权日：不复权当日 close 与前一日 close 差距大，
# 但前复权差距小
raw_chg = byd_raw['close'].pct_change()
# 涨幅 < -5% 但不是跌停（A股有跌停规则）
candidates = byd_raw[(raw_chg < -0.05) & (raw_chg > -0.11)]
print(f"找到疑似除权日 {len(candidates)} 个，第一个：")
print(candidates[['date', 'close']].head(1))

In [ ]:
# 看那个日期三种复权方式的对比
if len(candidates) > 0:
    ex_date = candidates['date'].iloc[0]
    print(f"除权日 {ex_date.date()} 三种复权对比：")
    print(f"  不复权 close: {byd_raw[byd_raw['date']==ex_date]['close'].iloc[0]:.2f}")
    print(f"  前复权 close: {byd_qfq[byd_qfq['date']==ex_date]['close'].iloc[0]:.2f}")
    print(f"  后复权 close: {byd_hfq[byd_hfq['date']==ex_date]['close'].iloc[0]:.2f}")

## 第 3 段：多股纵向合并 `pd.concat`

把三只股票的日线**纵向**堆起来（行变多），加 code 列区分。

In [ ]:
stocks = [("002594", "比亚迪"), ("002602", "世纪华通"), ("002624", "完美世界")]
frames = []
for code, name in stocks:
    df = get_stock_data(code)
    df = df.copy()
    df['code'] = code
    df['name'] = name
    frames.append(df)

long_df = pd.concat(frames, ignore_index=True)
print(f"三股纵向合并：{len(long_df)} 行 × {len(long_df.columns)} 列")
long_df.head()

## 第 4 段：多股横向对齐 `pivot` + `reindex`

把 close 列**横向**铺开，每只股票一列，日期为 index。
**关键**：日期必须对齐（停牌日要补齐）。

In [ ]:
# pivot 成 wide format
wide_close = long_df.pivot(index='date', columns='code', values='close')
print(f"对齐前 shape: {wide_close.shape}")
print(f"NaN 总数（说明有日期错位）: {wide_close.isna().sum().sum()}")
wide_close.head()

In [ ]:
# 用交易日历 reindex（用三股的并集作为完整日历）
full_dates = wide_close.index  # pd.concat 已隐式对齐
# 如果某只股票某天缺数据（停牌），用 ffill 填充
wide_close_filled = wide_close.sort_index().ffill()
print(f"填充后 NaN 总数: {wide_close_filled.isna().sum().sum()}")
wide_close_filled.head()

## 第 5 段：保存 panel 数据到 parquet

把对齐好的 wide format 存起来，L05+ 直接读用。

In [ ]:
from pathlib import Path
panel_path = Path("data/panel_three_stocks.parquet")
wide_close_filled.to_parquet(panel_path)
print(f"已保存：{panel_path} ({panel_path.stat().st_size // 1024} KB)")

# 验证读回
read_back = pd.read_parquet(panel_path)
print(f"读回 shape: {read_back.shape}")
read_back.head()

## 第 6 段：随堂小练

### 小练：5 股对齐 panel
拉取 5 只代码（比亚迪、世纪华通、完美世界、贵州茅台 600519、宁德时代 300750），
对齐成 wide format，存为 `data/panel_five_stocks.parquet`。

In [ ]:
# TODO: 你的代码（约 10 行）

## 第 7 段：课后练习 + 下节预告

### 📝 `exercises/ex04.py`
1. 写 `build_panel(codes: list[str], adjust='qfq') -> pd.DataFrame`，输入代码列表，返回对齐后的 wide close DataFrame
2. 写 `detect_suspicious_gap(df, threshold=0.15) -> list[tuple]`，检测不复权数据里"单日跌幅 > threshold 但不是跌停"的可疑除权日（return [(date, code, pct_change), ...]）
3. 对 5 股 panel 计算两两相关矩阵，用 seaborn 画热力图

### 🔮 下节 L05：收益率 + 涨停识别
本节的对齐 panel 是 L05 的输入。下节学 `pct_change / shift / diff / cumprod`，并真正识别"一字涨停"。

## 第 8 段：Jupyter tip 🔧
- `%time wide_close.corr()`：测整段代码（多行用 `%%time`）
- `df.info(memory_usage='deep')`：看真实内存占用
- `pd.set_option('display.max_rows', 100)`：调整显示行数
- 处理大表前先 `df = df.convert_dtypes()` 或手动 `astype('float32')` 降精度省内存